In [ ]:
import os
import time
import shutil
import warnings
import time

# Suppress future warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Create constraints file
!echo "numpy<2.0" > /kaggle/working/constraints.txt

# Set environment variable for the entire notebook
import os
os.environ['PIP_CONSTRAINT'] = '/kaggle/working/constraints.txt'

# OOM FIX: less CUDA allocator fragmentation (the failed run had 14.7 GB reserved but only 7.6 GB allocated).
# Used together with --disable-cuda-malloc below, otherwise ComfyUI appends backend:cudaMallocAsync and
# the setting is not honoured.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Pin numpy<2 FIRST, before anything in this kernel imports numpy/torch.
# (If numpy 2.x gets imported and is then downgraded on disk by later pip installs,
#  lazy-loaded submodules like numpy.random break with
#  "numpy.dtype size changed, may indicate binary incompatibility".)
!pip install -q "numpy<2.0"

# --------------------------
# 0. 清理旧进程
# --------------------------
print("清理旧进程...")
!pkill -f 'python main.py' || true
!pkill -f 'pinggy' || true

# --------------------------
# 0.5 检测和配置多GPU
# --------------------------
print("\n" + "="*60)
print("检测 GPU 配置...")
print("="*60)

# NOTE: use nvidia-smi instead of `import torch` here, so numpy/torch are NOT loaded
# into this kernel before the package installs below change them on disk.
import subprocess
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        text=True)
    gpu_lines = [l.strip() for l in out.strip().splitlines() if l.strip()]
    num_gpus = len(gpu_lines)
    print(f"✓ Detected {num_gpus} GPU(s)")
    for l in gpu_lines:
        print(f"  GPU {l}")
    if num_gpus > 1:
        os.environ['CUDA_VISIBLE_DEVICES'] = ','.join(map(str, range(num_gpus)))
except Exception as ex:
    print(f"✗ GPU detection failed: {ex}")

print("="*60)

# --------------------------
# 1. 系统依赖
# --------------------------
print("安装系统库...")
!apt-get update -qq
!apt-get install -y -qq libgl1 libglib2.0-0 ffmpeg

# --------------------------
# 2. 克隆 ComfyUI
# --------------------------
print("克隆 ComfyUI 仓库...")
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd ComfyUI
print(f"当前工作目录: {os.getcwd()}")
print("更新 ComfyUI...")
!git pull

# --------------------------
# 3. 安装 Python 库 (优化版本组合)
# --------------------------
print("安装额外依赖...")
!pip install -q mediapipe
!pip uninstall onnxruntime -y
!pip install onnxruntime-gpu

import onnxruntime as ort
print(ort.get_available_providers())
# Should show: ['CUDAExecutionProvider', 'CPUExecutionProvider']

# SAM-2 needs to be installed from git
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git

print("卸载不兼容库...")
!pip uninstall -y torch torchvision torchaudio xformers

print("安装最新 PyTorch nightly (WanVideoWrapper 要求 2.7.0.dev20250226+)...")
# !pip install --pre --upgrade --extra-index-url https://download.pytorch.org/whl/nightly/cu121 torch torchvision torchaudio
!pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu128

print("安装兼容的 xformers...")
!pip install -q xformers

print("安装 ComfyUI 其余依赖...")

# Downgrade numpy to avoid incompatibility with custom nodes

!pip install -q -r requirements.txt --no-deps
!pip install -q torchsde einops transformers safetensors aiohttp accelerate pyyaml opencv-python pillow scipy imageio[ffmpeg] moviepy huggingface_hub gguf ftfy
!pip install -q pydantic-settings python-dotenv

# --------------------------
# 4. 验证安装
# --------------------------
print("\n" + "="*50)
print("验证 PyTorch 和 xformers...")
print("="*50)
try:
    import torch
    import torchvision
    import xformers
    import xformers.ops
    
    print(f"✓ PyTorch: {torch.__version__}")
    print(f"✓ TorchVision: {torchvision.__version__}")
    print(f"✓ xformers: {xformers.__version__}")
    print(f"✓ CUDA 可用: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        num_gpus = torch.cuda.device_count()
        print(f"✓ GPU 数量: {num_gpus}")
        for i in range(num_gpus):
            print(f"  - GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"✓ CUDA 版本: {torch.version.cuda}")
    
    # 验证 RMSNorm 可用性
    if hasattr(torch.nn, 'RMSNorm'):
        print("✓ RMSNorm 可用 (WanVideoWrapper 将正常工作)")
    else:
        print("✗ RMSNorm 不可用")
        
    print("="*50)
except Exception as e:
    print(f"✗ 验证失败: {e}")
    print("="*50)

# --------------------------
# 5. Download all models automatically (no Kaggle datasets needed)
# --------------------------
# Files are downloaded from Hugging Face into a scratch folder (NOT /kaggle/working,
# so they don't count toward the 20 GB saved-output limit) and then symlinked
# into ComfyUI/models/* with the exact filenames the workflow expects.
#
# Optional: add a Kaggle secret named HF_TOKEN (Add-ons > Secrets) for faster /
# rate-limit-free downloads. Everything here is public, so it is NOT required.

print("\n" + "=" * 60)
print("Downloading models from Hugging Face...")
print("=" * 60)

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"   # faster downloads with the default hf_xet backend

from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---- Choose the GGUF quantization here ("Q6_K" for quality, "Q3_K_M" for low memory) ----
GGUF_QUANT = "Q6_K"

# Optional: HF token from Kaggle secrets (silently ignored if not set)
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

# Scratch download location
MODEL_CACHE = "/kaggle/temp/wan_models"
try:
    os.makedirs(MODEL_CACHE, exist_ok=True)
except Exception:
    MODEL_CACHE = "/kaggle/working/wan_models_cache"
    os.makedirs(MODEL_CACHE, exist_ok=True)
print(f"Download cache: {MODEL_CACHE}")

# ComfyUI model folders
model_dirs = {
    "diffusion_models": "./models/diffusion_models",
    "unet": "./models/unet",
    "text_encoders": "./models/text_encoders",
    "vae": "./models/vae",
    "loras": "./models/loras",
    "clip_vision": "./models/clip_vision",
}
for dir_path in model_dirs.values():
    os.makedirs(dir_path, exist_ok=True)

# (repo_id, path inside repo, ComfyUI target folder)
MODEL_FILES = [
    # GGUF diffusion model
    ("QuantStack/Wan2.2-Animate-14B-GGUF",
     f"Wan2.2-Animate-14B-{GGUF_QUANT}.gguf",
     "./models/unet"),
    # Text encoder
    ("Comfy-Org/Wan_2.2_ComfyUI_Repackaged",
     "split_files/text_encoders/umt5_xxl_fp16.safetensors",
     "./models/text_encoders"),
    # VAE
    ("Kijai/WanVideo_comfy",
     "Wan2_1_VAE_bf16.safetensors",
     "./models/vae"),
    # LoRAs
    ("Kijai/WanVideo_comfy",
     "LoRAs/Wan22_relight/WanAnimate_relight_lora_fp16.safetensors",
     "./models/loras"),
    ("Kijai/WanVideo_comfy",
     "Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors",
     "./models/loras"),
    # CLIP Vision
    ("Comfy-Org/Wan_2.1_ComfyUI_repackaged",
     "split_files/clip_vision/clip_vision_h.safetensors",
     "./models/clip_vision"),
]

def download_and_link(repo_id, repo_path, target_dir):
    filename = os.path.basename(repo_path)
    link_path = os.path.abspath(os.path.join(target_dir, filename))

    # Skip if a valid file/link already exists (e.g. re-running the cell)
    if os.path.exists(link_path) and os.path.getsize(link_path) > 0:
        return filename, "already present"

    last_err = None
    for attempt in range(1, 4):  # retry up to 3 times
        try:
            local_file = hf_hub_download(
                repo_id=repo_id,
                filename=repo_path,
                local_dir=os.path.join(MODEL_CACHE, repo_id.replace("/", "__")),
                token=HF_TOKEN,
            )
            break
        except Exception as e:
            last_err = e
            print(f"  ! {filename}: attempt {attempt}/3 failed: {e}")
            time.sleep(5 * attempt)
    else:
        raise RuntimeError(f"Could not download {filename} from {repo_id}: {last_err}")

    # Replace stale link/file, then symlink under the flat filename ComfyUI expects
    if os.path.lexists(link_path):
        os.remove(link_path)
    os.symlink(os.path.abspath(local_file), link_path)
    return filename, "downloaded + linked"

# Download in parallel (3 at a time is plenty on Kaggle bandwidth)
failed = []
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(download_and_link, *spec): spec for spec in MODEL_FILES}
    for fut in as_completed(futures):
        repo_id, repo_path, _ = futures[fut]
        try:
            name, status = fut.result()
            print(f"  ✓ {name}: {status}")
        except Exception as e:
            failed.append(repo_path)
            print(f"  ✗ {repo_path}: {e}")

if failed:
    print("\n⚠ Some models failed to download:", failed)
    print("  Re-run this cell; finished files are skipped automatically.")

# ---------- Optional: download the API workflow JSON for the Gradio app ----------
# Paste a direct/raw URL to "WAN2.2 Animate 12GB-VRAM-API.json" here to have it
# auto-loaded. Leave empty to upload the workflow manually in the Gradio UI.
WORKFLOW_URL = ""
WORKFLOW_DIR = "/kaggle/working/workflows"
os.makedirs(WORKFLOW_DIR, exist_ok=True)
if WORKFLOW_URL:
    try:
        wf_dest = os.path.join(WORKFLOW_DIR, "WAN2.2 Animate 12GB-VRAM-API.json")
        !wget -q -O "{wf_dest}" "{WORKFLOW_URL}"
        print(f"  ✓ Workflow downloaded to {wf_dest}")
    except Exception as e:
        print(f"  ⚠ Workflow download failed: {e}")

# Summary + verification
print("\n" + "=" * 60)
print("Model summary")
print("=" * 60)
for name, path in model_dirs.items():
    files = sorted(os.listdir(path)) if os.path.isdir(path) else []
    if not files:
        continue
    print(f"\n{name}:")
    for f in files:
        full = os.path.join(path, f)
        if os.path.islink(full) and not os.path.exists(full):
            print(f"  ✗ BROKEN LINK: {f}")
        else:
            size_gb = os.path.getsize(full) / 1024**3
            print(f"  ✓ {f}  ({size_gb:.2f} GB)")
!df -h /kaggle/temp /kaggle/working 2>/dev/null

# --------------------------
# 6. 安装所有必需插件
# --------------------------
print("\n" + "="*60)
print("安装 ComfyUI 插件...")
print("="*60)

if os.path.exists("/kaggle/input/wan-animate-custom-nodes"):
    src = "/kaggle/input/wan-animate-custom-nodes/custom_nodes"
    dest = "/kaggle/working/ComfyUI/custom_nodes"
    
    # Copy folder directly
    shutil.copytree(src, dest, dirs_exist_ok=True)
    
    print("Custom nodes copied")

%cd custom_nodes

# Complete list of plugins including Kijai workflow requirements
all_plugins = {
    # Core plugins
    "ComfyUI-Manager": "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "ComfyUI-GGUF": "https://github.com/city96/ComfyUI-GGUF.git",
    "ComfyUI-WanVideoWrapper": "https://github.com/kijai/ComfyUI-WanVideoWrapper.git",
    
    # Kijai workflow requirements
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "comfyui_controlnet_aux": "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    "ComfyUI-VideoHelperSuite": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "ComfyUI-segment-anything-2": "https://github.com/kijai/ComfyUI-segment-anything-2.git",
    "ComfyUI-Advanced-ControlNet": "https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet.git",
    "comfyui-tooling-nodes": "https://github.com/Acly/comfyui-tooling-nodes.git",
    "ComfyUI_essentials": "https://github.com/cubiq/ComfyUI_essentials.git",
    
    # Additional required nodes
    "cg-use-everywhere": "https://github.com/chrisgoringe/cg-use-everywhere.git",
    "was-node-suite-comfyui": "https://github.com/WASasquatch/was-node-suite-comfyui.git",
    "ComfyUI-Custom-Scripts": "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git",
    "ComfyUI-Frame-Interpolation": "https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git",
    "rgthree-comfy": "https://github.com/rgthree/rgthree-comfy.git",
    "ComfyMath": "https://github.com/evanspearman/ComfyMath.git",
    "ComfyUI-Easy-Use": "https://github.com/yolain/ComfyUI-Easy-Use.git",
    "comfyui-mixlab-nodes": "https://github.com/MixLabPro/comfyui-mixlab-nodes.git"
}

for plugin_name, plugin_url in all_plugins.items():
    print(f"\n{'='*60}")
    print(f"处理: {plugin_name}")
    print(f"{'='*60}")
    
    if not os.path.exists(plugin_name):
        print(f"  → 克隆 {plugin_name}...")
        !git clone {plugin_url}
        
        # Install requirements
        req_file = f"{plugin_name}/requirements.txt"
        if os.path.exists(req_file):
            print(f"  → 安装依赖...")
            !pip install -q -r {req_file}
            
        print(f"  ✓ {plugin_name} 完成")
    # else:
    #     print(f"  → {plugin_name} 已存在，更新中...")
    #     %cd {plugin_name}
    #     !git pull -q
        
    #     # Update requirements
    #     if os.path.exists("requirements.txt"):
    #         !pip install -q -r requirements.txt
    #     %cd ..
    #     print(f"  ✓ {plugin_name} 更新完成")

%cd ..
print("\n" + "="*60)
print(f"✓ 所有插件安装完成！")
print(f"当前工作目录: {os.getcwd()}")
print("="*60)

# --------------------------
# 6.5 配置 ComfyUI 多GPU支持
# --------------------------
print("\n" + "="*60)
print("配置 ComfyUI 多GPU支持...")
print("="*60)

# Remove or fix the extra_model_paths.yaml file to prevent startup errors
config_file = "extra_model_paths.yaml"
if os.path.exists(config_file):
    os.remove(config_file)
    print(f"✓ 已删除旧配置文件: {config_file}")

# ComfyUI will use default paths, which works better with multi-GPU
print("✓ 使用默认配置 (自动多GPU支持)")
print("="*60)

# Suppress warnings in environment
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTHONWARNINGS'] = 'ignore::FutureWarning'

# Change to ComfyUI directory
%cd /kaggle/working/ComfyUI

print("\n🚀 启动 ComfyUI 服务器 (多GPU模式)...")

# Check GPU count one more time
try:
    import torch
    num_gpus = torch.cuda.device_count()
    if num_gpus > 1:
        print(f"✓ 启用 {num_gpus} GPU 模式")
        print(f"  CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'all')}")
    print()
except:
    pass

# --------------------------
# 7. 启动 ComfyUI 作为后台进程
# --------------------------
import subprocess
import time
import requests
import sys

print("\n" + "="*60)
print("🚀 启动 ComfyUI 服务器 (后台进程模式)...")
print("="*60)

# Ensure we're in the ComfyUI directory
os.chdir('/kaggle/working/ComfyUI')
print(f"当前目录: {os.getcwd()}")

# Configure launch arguments
comfy_args = [
    sys.executable,  # Use current Python interpreter
    'main.py',
    '--listen', '0.0.0.0',
    '--port', '8188',
    '--preview-method', 'auto',
    '--cache-none',
    '--disable-cuda-malloc',   # OOM FIX: use the native allocator so expandable_segments applies
    '--output-directory', '/kaggle/working/ComfyUI/output'  # Add output path
]

# Add multi-GPU support if available
try:
    import torch
    num_gpus = torch.cuda.device_count()
    if num_gpus > 1:
        print(f"✓ 配置多GPU模式 ({num_gpus} GPUs)")
        # ComfyUI will automatically use all available GPUs
except:
    pass

# Start ComfyUI as background subprocess
print(f"\n启动命令: {' '.join(comfy_args)}")
print("="*60)

comfy_env = os.environ.copy()
comfy_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

comfy_process = subprocess.Popen(
    comfy_args,
    env=comfy_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd='/kaggle/working/ComfyUI',
    bufsize=1,
    universal_newlines=True
)

print(f"✓ ComfyUI 进程已启动")
print(f"  PID: {comfy_process.pid}")
print(f"  状态: {'运行中' if comfy_process.poll() is None else '已停止'}")

# --------------------------
# 8. 等待 ComfyUI 就绪
# --------------------------
def wait_for_comfyui(max_attempts=60, delay=2):
    """等待 ComfyUI 服务器就绪"""
    print(f"\n⏳ 等待 ComfyUI 启动 (最多 {max_attempts * delay} 秒)...")
    print("-" * 60)
    
    for i in range(max_attempts):
        # Check if process is still running
        if comfy_process.poll() is not None:
            print(f"\n✗ ComfyUI 进程意外终止!")
            print("\n最后的输出:")
            if comfy_process.stdout:
                print(comfy_process.stdout.read())
            if comfy_process.stderr:
                print("\n错误输出:")
                print(comfy_process.stderr.read())
            return False
        
        try:
            # Try to connect to ComfyUI API
            response = requests.get(
                "http://127.0.0.1:8188/system_stats",
                timeout=2
            )
            
            if response.status_code == 200:
                print(f"\n✓ ComfyUI 已就绪! (耗时 {(i+1) * delay} 秒)")
                print("-" * 60)
                
                # Print system stats
                try:
                    stats = response.json()
                    print("\n系统状态:")
                    if 'system' in stats:
                        sys_info = stats['system']
                        print(f"  • OS: {sys_info.get('os', 'N/A')}")
                        print(f"  • Python: {sys_info.get('python_version', 'N/A')}")
                    
                    if 'devices' in stats:
                        devices = stats['devices']
                        print(f"\n  • GPU 设备:")
                        for i, device in enumerate(devices):
                            print(f"    - GPU {i}: {device.get('name', 'N/A')}")
                            vram_total = device.get('vram_total', 0) / (1024**3)
                            print(f"      VRAM: {vram_total:.2f} GB")
                except:
                    pass
                
                print("="*60)
                return True
                
        except requests.exceptions.RequestException:
            pass
        except Exception as e:
            pass
        
        # Progress indicator
        time.sleep(delay)
        if (i + 1) % 5 == 0:
            print(f"  尝试 {i+1}/{max_attempts} - 仍在启动中...")
    
    print(f"\n⚠️ 超时: ComfyUI 未能在 {max_attempts * delay} 秒内启动")
    print("\n检查进程状态...")
    print(f"  进程状态: {'运行中' if comfy_process.poll() is None else '已停止'}")
    
    return False

# Wait for ComfyUI to be ready
if wait_for_comfyui():
    print("\n🎉 ComfyUI 服务器运行正常!")
    print(f"    → 本地访问: http://127.0.0.1:8188")
    print(f"    → 进程 PID: {comfy_process.pid}")
    print("\n✓ 可以继续运行下一个单元格启动 Gradio")
else:
    print("\n⚠️  ComfyUI 启动可能遇到问题")
    print("    → 建议检查下面的日志输出")
    print("    → 或运行调试单元格查看详细错误")

print("="*60)

# --------------------------
# 9. 可选: 实时查看 ComfyUI 日志
# --------------------------
# 如果需要查看实时日志，可以运行这个单元格:

import threading

def stream_output(pipe, label):
    '''实时打印进程输出'''
    for line in iter(pipe.readline, ''):
        if line:
            print(f"[{label}] {line.rstrip()}")

# 创建日志线程
stdout_thread = threading.Thread(
    target=stream_output, 
    args=(comfy_process.stdout, "STDOUT"),
    daemon=True
)
stderr_thread = threading.Thread(
    target=stream_output,
    args=(comfy_process.stderr, "STDERR"),
    daemon=True
)

stdout_thread.start()
stderr_thread.start()

print("✓ 日志监控已启动 (后台运行)")

# --------------------------
# 10. 调试单元格 (可选)
# --------------------------
# 如果 ComfyUI 启动失败，运行这个来查看详细信息:

print("="*60)
print("ComfyUI 进程诊断")
print("="*60)

print(f"\n进程 PID: {comfy_process.pid}")
print(f"进程状态: {comfy_process.poll()}")
print(f"运行中: {comfy_process.poll() is None}")

if comfy_process.poll() is not None:
    print("\n⚠️  进程已停止")
    
    print("\n标准输出:")
    print("-" * 60)
    if comfy_process.stdout:
        stdout = comfy_process.stdout.read()
        print(stdout if stdout else "(无输出)")
    
    print("\n标准错误:")
    print("-" * 60)
    if comfy_process.stderr:
        stderr = comfy_process.stderr.read()
        print(stderr if stderr else "(无错误)")
else:
    print("\n✓ 进程仍在运行")
    
    # Test API connection
    try:
        response = requests.get("http://127.0.0.1:8188/system_stats", timeout=2)
        print(f"\n✓ API 响应: {response.status_code}")
        print(response.json())
    except Exception as e:
        print(f"\n✗ API 连接失败: {e}")

print("="*60)

In [ ]:
%%writefile /kaggle/working/gradio_app.py
# GRADIO + COMFYUI API (runs as its own process, see next cell)

import json
import copy
import glob
import requests
import gradio as gr
import websocket
import uuid
import urllib.request
import urllib.parse
from PIL import Image
import io
import os
import time

class ComfyUIAPI:
    def __init__(self, server_address="127.0.0.1:8188"):
        self.server_address = server_address
        self.client_id = str(uuid.uuid4())
        
    def queue_prompt(self, prompt):
        """Queue a prompt to ComfyUI"""
        p = {"prompt": prompt, "client_id": self.client_id}
        data = json.dumps(p).encode('utf-8')
        req = urllib.request.Request(f"http://{self.server_address}/prompt", data=data)
        return json.loads(urllib.request.urlopen(req).read())
    
    def upload_image(self, image_path, subfolder="", overwrite=False):
        """Upload an image to ComfyUI"""
        with open(image_path, 'rb') as f:
            files = {'image': f}
            data = {
                'subfolder': subfolder,
                'overwrite': str(overwrite).lower()
            }
            response = requests.post(
                f"http://{self.server_address}/upload/image",
                files=files,
                data=data
            )
        return response.json()
    
    def upload_video(self, video_path, subfolder="", overwrite=True):
        """Upload a video to ComfyUI"""
        with open(video_path, 'rb') as f:
            files = {'image': (os.path.basename(video_path), f, 'video/mp4')}
            data = {
                'subfolder': subfolder,
                'overwrite': str(overwrite).lower(),
                'type': 'input'
            }
            response = requests.post(
                f"http://{self.server_address}/upload/image",
                files=files,
                data=data
            )
        return response.json()
    
    def free_memory(self):
        """Ask ComfyUI to unload models and release cached VRAM before a new run."""
        try:
            requests.post(
                f"http://{self.server_address}/free",
                json={"unload_models": True, "free_memory": True},
                timeout=30,
            )
        except Exception:
            pass
    
    def get_history(self, prompt_id):
        """Get the execution history for a prompt"""
        req = urllib.request.Request(f"http://{self.server_address}/history/{prompt_id}")
        return json.loads(urllib.request.urlopen(req).read())
    
    def get_image(self, filename, subfolder, folder_type):
        """Download an image from ComfyUI"""
        data = {"filename": filename, "subfolder": subfolder, "type": folder_type}
        url_values = urllib.parse.urlencode(data)
        url = f"http://{self.server_address}/view?{url_values}"
        with urllib.request.urlopen(url) as response:
            return response.read()
    
    def get_video(self, filename, subfolder, folder_type, save_path):
        """Download a video from ComfyUI and save it locally"""
        data = {"filename": filename, "subfolder": subfolder, "type": folder_type}
        url_values = urllib.parse.urlencode(data)
        url = f"http://{self.server_address}/view?{url_values}"
        
        # Download video
        with urllib.request.urlopen(url) as response:
            video_data = response.read()
        
        # Save to file
        with open(save_path, 'wb') as f:
            f.write(video_data)
        
        return save_path
    
    def track_progress(self, prompt_id, progress_callback=None):
        """Track workflow execution progress via WebSocket"""
        ws = websocket.WebSocket()
        ws.connect(f"ws://{self.server_address}/ws?clientId={self.client_id}")
        
        while True:
            out = ws.recv()
            if isinstance(out, str):
                message = json.loads(out)
                
                if message['type'] == 'progress' and progress_callback:
                    data = message['data']
                    progress_callback(data.get('value', 0), data.get('max', 100))
                
                if message['type'] == 'execution_error':
                    data = message['data']
                    if data.get('prompt_id') == prompt_id:
                        ws.close()
                        raise RuntimeError(
                            f"ComfyUI error in node {data.get('node_id')} ({data.get('node_type')}): "
                            f"{data.get('exception_type')}: {data.get('exception_message')}"
                        )
                
                if message['type'] == 'execution_interrupted':
                    if message['data'].get('prompt_id') == prompt_id:
                        ws.close()
                        raise RuntimeError("Execution was interrupted.")
                
                if message['type'] == 'executing':
                    data = message['data']
                    if data['node'] is None and data['prompt_id'] == prompt_id:
                        break
        ws.close()

class WanVideoGradioApp:
    def __init__(self):
        self.api = ComfyUIAPI()
        self.workflow = None
        self.temp_dir = "gradio_outputs"
        os.makedirs(self.temp_dir, exist_ok=True)
        self.auto_load_status = self._try_auto_load_workflow()
        
    def _try_auto_load_workflow(self):
        """Try to automatically load workflow from Kaggle input directory"""
        kaggle_input_paths = [
            "/kaggle/working/workflows",            # downloaded by the setup cell (WORKFLOW_URL)
            "/kaggle/input/wan-video-workflow",     # optional: attached dataset (legacy)
        ]
        
        workflow_filenames = [
            "WAN2.2 Animate 12GB-VRAM-API.json",
        ]
        
        # Search for workflow file
        for base_path in kaggle_input_paths:
            if not os.path.exists(base_path):
                continue
                
            # Check base path
            for filename in workflow_filenames:
                filepath = os.path.join(base_path, filename)
                if os.path.exists(filepath):
                    try:
                        with open(filepath, 'r') as f:
                            self.workflow = json.load(f)
                        return f"✓ Auto-loaded workflow from: {filepath}\nFound {len(self.workflow)} nodes. Ready to process videos."
                    except Exception as e:
                        return f"⚠ Found workflow at {filepath} but failed to load: {str(e)}"
            
            # Check subdirectories
            try:
                for root, dirs, files in os.walk(base_path):
                    for filename in workflow_filenames:
                        if filename in files:
                            filepath = os.path.join(root, filename)
                            try:
                                with open(filepath, 'r') as f:
                                    self.workflow = json.load(f)
                                return f"✓ Auto-loaded workflow from: {filepath}\nFound {len(self.workflow)} nodes. Ready to process videos."
                            except Exception as e:
                                return f"⚠ Found workflow at {filepath} but failed to load: {str(e)}"
            except Exception:
                pass
        
        return "⚠ No workflow found. Set WORKFLOW_URL in the setup cell, or upload the workflow JSON manually below."
        
    def _patch_workflow(self, wf):
        """Fix API-format workflows exported with older node versions, without needing node ids."""
        notes = []
        for nid, node in wf.items():
            if not isinstance(node, dict):
                continue
            ins = node.get("inputs", {})
            ctype = node.get("class_type", "")
            # Newer ComfyUI-Frame-Interpolation RIFE VFI requires these three inputs
            if ctype.startswith("RIFE VFI"):
                added = []
                for k, v in (("torch_compile", False), ("batch_size", 1), ("dtype", "float16")):
                    if k not in ins:
                        ins[k] = v
                        added.append(k)
                if added:
                    notes.append(f"🔧 RIFE VFI {nid}: added missing inputs {added}")
            # umt5_xxl_fp16 is ~11 GB and stays staged in VRAM on a T4 -> run the text encoder on CPU
            if ctype == "CLIPLoader" and ins.get("type") == "wan" and ins.get("device") == "default":
                ins["device"] = "cpu"
                notes.append(f"🔧 CLIPLoader {nid}: text encoder set to CPU (saves ~11 GB VRAM)")
            # Use whichever Wan2.2-Animate GGUF was actually downloaded
            if ctype == "WanVideoModelLoader":
                unet_dir = "/kaggle/working/ComfyUI/models/unet"
                name = ins.get("model", "")
                if name and not os.path.exists(os.path.join(unet_dir, name)):
                    found = sorted(glob.glob(os.path.join(unet_dir, "Wan2.2-Animate-14B-*.gguf")))
                    if found:
                        ins["model"] = os.path.basename(found[0])
                        notes.append(f"🔧 Model {name} not found, using {ins['model']}")
            # OOM FIX: always turn on VAE tiling wherever the workflow exposes it
            # (the OOM happened in the non-tiled VAE decode after the first sampling window)
            for k in ("tiled_vae", "enable_vae_tiling"):
                if k in ins and isinstance(ins[k], bool) and ins[k] is not True:
                    ins[k] = True
                    notes.append(f"🔧 {ctype} {nid}: enabled {k}")
            # torch.compile (inductor) gives little on a T4 and costs extra VRAM + a long warm-up
            if ctype == "WanVideoModelLoader":
                if "compile_args" in ins:
                    del ins["compile_args"]
                    notes.append(f"🔧 WanVideoModelLoader {nid}: torch.compile disabled")
                # fp16_fast can be unstable on Turing (T4); plain fp16 is safe
                if ins.get("base_precision") == "fp16_fast":
                    ins["base_precision"] = "fp16"
                    notes.append(f"🔧 WanVideoModelLoader {nid}: base_precision fp16_fast -> fp16 (T4)")
        return notes

    def load_workflow(self, workflow_file):
        """Load the WanVideo workflow JSON file"""
        try:
            if workflow_file is None:
                return "⚠ Please upload a workflow file"
                
            with open(workflow_file.name, 'r') as f:
                self.workflow = json.load(f)
            return f"✓ Workflow loaded successfully! Found {len(self.workflow)} nodes.\nReady to process videos."
        except Exception as e:
            return f"✗ Error loading workflow: {str(e)}"
    
    def run_workflow(self, prompt_text, negative_prompt, input_image, input_video, 
                    length_seconds, framerate, resolution, interpolation_factor, seed, fws_divisor=2):
        """Execute the WanVideo workflow with provided inputs"""
        if not self.workflow:
            return "⚠ Please load a workflow first!", None, None, None
        
        try:
            status_updates = []
            
            # Upload image
            if input_image:
                status_updates.append("📤 Uploading image...")
                img_result = self.api.upload_image(input_image)
                self.workflow["64"]["inputs"]["image"] = img_result['name']
                status_updates.append(f"✓ Image uploaded: {img_result['name']}")
            
            # Upload video
            if input_video:
                status_updates.append("📤 Uploading video...")
                vid_result = self.api.upload_video(input_video)
                self.workflow["13"]["inputs"]["video"] = vid_result['name']
                status_updates.append(f"✓ Video uploaded: {vid_result['name']}")
            
            # Update text prompts
            if prompt_text:
                self.workflow["124"]["inputs"]["text"] = prompt_text
            if negative_prompt:
                self.workflow["122"]["inputs"]["text"] = negative_prompt
            
            # Update parameters
            self.workflow["28"]["inputs"]["value"] = float(length_seconds)
            self.workflow["30"]["inputs"]["value"] = float(framerate)
            self.workflow["32"]["inputs"]["value"] = int(resolution)
            self.workflow["29"]["inputs"]["value"] = int(interpolation_factor)
            self.workflow["90"]["inputs"]["seed"] = int(seed)
            self.workflow["161"]["inputs"]["value"] = max(1, int(fws_divisor))  # frame-window divisor
            
            status_updates.append("⚙️ Starting workflow execution...")
            
            # Patch the workflow for newer node versions / low VRAM, then queue a copy
            self.api.free_memory()
            wf = copy.deepcopy(self.workflow)
            for note in self._patch_workflow(wf):
                status_updates.append(note)
            result = self.api.queue_prompt(wf)
            prompt_id = result['prompt_id']
            status_updates.append(f"✓ Workflow queued (ID: {prompt_id[:8]}...)")
            
            # Track progress
            status_updates.append("🔄 Processing... This may take several minutes.")
            self.api.track_progress(prompt_id)
            
            status_updates.append("✓ Workflow completed!")
            status_updates.append("📦 Retrieving outputs...")
            
            # Get output
            time.sleep(2)  # Wait for files to be written
            history = self.api.get_history(prompt_id)[prompt_id]
            
            # Get output images/videos
            output_images = []
            output_videos = []
            video_by_node = {}
            output_info = []
            
            for node_id in history['outputs']:
                node_output = history['outputs'][node_id]
                
                # Get images
                if 'images' in node_output:
                    for image in node_output['images']:
                        try:
                            image_data = self.api.get_image(
                                image['filename'], 
                                image['subfolder'], 
                                image['type']
                            )
                            img = Image.open(io.BytesIO(image_data))
                            output_images.append(img)
                            output_info.append(f"Image: {image['filename']}")
                        except Exception as e:
                            status_updates.append(f"⚠ Could not load image: {str(e)}")
                
                # Get videos (including gifs)
                if 'gifs' in node_output:
                    for idx, video in enumerate(node_output['gifs']):
                        try:
                            # Save video locally
                            video_filename = f"output_{prompt_id[:8]}_{node_id}_{idx}.mp4"
                            video_path = os.path.join(self.temp_dir, video_filename)
                            
                            self.api.get_video(
                                video['filename'],
                                video['subfolder'],
                                video['type'],
                                video_path
                            )
                            
                            output_videos.append(video_path)
                            video_by_node[str(node_id)] = video_path
                            output_info.append(f"Video: {video['filename']} → {video_filename}")
                            status_updates.append(f"✓ Downloaded: {video['filename']}")
                        except Exception as e:
                            status_updates.append(f"⚠ Could not download video: {str(e)}")
            
            status_updates.append(f"✓ Found {len(output_info)} outputs")
            
            final_status = "\n".join(status_updates)
            
            # Return outputs
            preview_image = output_images[0] if output_images else None
            # 106 = base animation, 178 = interpolated (52 is only the pose preview video)
            base_video = video_by_node.get("106") or (output_videos[0] if output_videos else None)
            interpolated_video = video_by_node.get("178")
            
            return final_status, preview_image, base_video, interpolated_video
                
        except Exception as e:
            error_msg = f"✗ Error executing workflow: {str(e)}"
            if "OutOfMemory" in str(e) or "Allocation on device" in str(e) or "out of memory" in str(e).lower():
                error_msg += ("\n\n💡 Out of VRAM. Try: lower Resolution (e.g. 320-384), shorter Length, "
                              "or a higher Window divisor, then run again.")
            import traceback
            error_msg += f"\n\nDetails:\n{traceback.format_exc()}"
            return error_msg, None, None, None

# Initialize the app
app = WanVideoGradioApp()

# Create Gradio interface
with gr.Blocks(title="WanVideo ComfyUI Controller", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎬 WanVideo Animation Controller
    ### Control WanVideo workflow via ComfyUI API with Gradio
    """)
    
    with gr.Row():
        workflow_file = gr.File(
            label="📁 Upload Workflow JSON (wan_2_2_animate.json)",
            file_types=[".json"]
        )
        load_btn = gr.Button("Load Workflow", variant="primary", size="sm")
    
    workflow_status = gr.Textbox(
        label="Workflow Status",
        interactive=False,
        lines=2,
        value=app.auto_load_status
    )
    
    gr.Markdown("---")
    gr.Markdown("### 🎨 Input Configuration")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("**📝 Text Inputs**")
            prompt_text = gr.Textbox(
                label="Positive Prompt",
                placeholder="Describe the animation you want...",
                lines=3,
                value="Shrek is dancing."
            )
            negative_prompt = gr.Textbox(
                label="Negative Prompt",
                placeholder="What to avoid in the animation...",
                lines=3,
                value="过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走, tattoo,"
            )
        
        with gr.Column():
            gr.Markdown("**🖼️ Media Inputs**")
            input_image = gr.Image(
                label="Reference Image",
                type="filepath",
                height=200
            )
            input_video = gr.Video(
                label="Input Video (Optional)",
                height=200
            )
    
    gr.Markdown("### ⚙️ Generation Parameters")
    
    with gr.Row():
        length_seconds = gr.Slider(
            minimum=1,
            maximum=30,
            value=5,
            step=1,
            label="🎬 Length (Seconds)"
        )
        framerate = gr.Slider(
            minimum=8,
            maximum=60,
            value=16,
            step=1,
            label="🎦 Framerate (FPS)"
        )
    
    with gr.Row():
        resolution = gr.Slider(
            minimum=256,
            maximum=1024,
            value=384,
            step=16,
            label="🔲 Resolution (Width)"
        )
        interpolation_factor = gr.Slider(
            minimum=1,
            maximum=4,
            value=2,
            step=1,
            label="🔀 Interpolation Factor"
        )
    
    with gr.Row():
        seed = gr.Number(
            label="🎲 Seed",
            value=777,
            precision=0
        )
        fws_divisor = gr.Slider(
            minimum=1,
            maximum=6,
            value=3,
            step=1,
            label="🧩 Window divisor (higher = smaller windows = less VRAM)"
        )
    
    run_btn = gr.Button("▶ Generate Animation", variant="primary", size="lg")
    
    gr.Markdown("---")
    gr.Markdown("### 📊 Results")
    
    execution_status = gr.Textbox(
        label="Execution Log",
        interactive=False,
        lines=8
    )
    
    with gr.Row():
        with gr.Column():
            output_image = gr.Image(
                label="Preview Frame"
            )
        with gr.Column():
            base_video_output = gr.Video(
                label="Base Animation"
            )
    
    interpolated_video_output = gr.Video(
        label="Interpolated Animation (High FPS)"
    )
    
    gr.Markdown("""
    ---
    ### 📖 Instructions:
    
    1. **Workflow Loading**: 
       - The app automatically searches for workflow files in `/kaggle/input/`
       - If not found, upload the `wan_2_2_animate.json` file manually
    2. **Set Prompts**: Describe the animation you want to create
    3. **Upload Media**: 
       - **Required**: Reference image (character/subject)
       - **Optional**: Input video for motion reference
    4. **Adjust Parameters**: 
       - Length: Duration in seconds
       - Framerate: Higher = smoother but slower to generate
       - Resolution: Output width (height auto-calculated)
       - Interpolation: Multiplier for frame interpolation (2 = double frames)
    5. **Generate**: Click "Generate Animation" and wait
    
    **Outputs**: 
    - Base Animation: Original generated video at specified FPS
    - Interpolated Animation: Enhanced version with interpolated frames for smoother motion
    
    **Note**: Make sure ComfyUI is running on `127.0.0.1:8188` with all required models loaded.
    """)
    
    # Event handlers
    load_btn.click(
        fn=app.load_workflow,
        inputs=[workflow_file],
        outputs=[workflow_status]
    )
    
    run_btn.click(
        fn=app.run_workflow,
        inputs=[
            prompt_text,
            negative_prompt,
            input_image,
            input_video,
            length_seconds,
            framerate,
            resolution,
            interpolation_factor,
            seed,
            fws_divisor
        ],
        outputs=[execution_status, output_image, base_video_output, interpolated_video_output]
    )
    
    # Load example
    gr.Examples(
        examples=[
            ["A person dancing energetically", "static, blurry, low quality", 5, 16, 384, 2, 777],
            ["Character walking forward confidently", "overexposed, static, ugly", 5, 16, 400, 2, 123],
        ],
        inputs=[prompt_text, negative_prompt, length_seconds, framerate, resolution, interpolation_factor, seed],
    )


demo.launch(share=True)


In [ ]:
import os, sys, subprocess

# 1) Install in a normal shell (does not touch this kernel's already-loaded numpy)
!pip install -q gradio requests websocket-client

# 2) Check the imports in a FRESH interpreter. The kernel itself loaded numpy early in the
#    setup cell, and later pip installs changed numpy on disk -> importing pandas/gradio
#    inside this kernel fails with "numpy.dtype size changed". A new process doesn't have that problem.
check = [sys.executable, "-c", "import numpy.random, pandas, gradio; print('gradio', gradio.__version__, '| numpy', numpy.__version__)"]
r = subprocess.run(check, capture_output=True, text=True)
if r.returncode != 0:
    print("Import check failed, repairing numpy/pandas...\n", r.stderr[-800:])
    env = {k: v for k, v in os.environ.items() if k != "PIP_CONSTRAINT"}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "numpy", "pandas"], env=env)
    r = subprocess.run(check, capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-800:])

# 3) Launch the Gradio app as a separate process and stream its output (keeps the notebook alive)
proc = subprocess.Popen(
    [sys.executable, "-u", "/kaggle/working/gradio_app.py"],
    cwd="/kaggle/working",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
print("Gradio started (PID %d). The public URL will appear below..." % proc.pid)
for line in proc.stdout:
    print(line, end="", flush=True)
print("\nGradio process exited with code", proc.wait())
